This notebook allows the user to change defacing parameters to see impacts on the performance. Other than dependencies in `requirements.txt`, also install ipykernel, ipympl and ipywidgets.

In [1]:
from raw_deface_opt import run_raw_deface
nc, data, dataID, og_image, og_image_rsos = run_raw_deface('config2.json') # run raw_deface and load data

Data has been successfully loaded
Using predefined masks
Mask has been successfully loaded
Nifti file successfully saved
Nifti file successfully saved
Masks have been prepared
Brain and face covariance matrices have been computed
Eigenvectors computed
The top nv eigenvectors contain the first 60 eigenvectors
BRAIN SIGNAL RETAINED:93.73104120044837
FACE SIGNAL RETAINED:6.16798871329835
Virtual coils successfully formed
Nifti file successfully saved
Mask saved successfully


In [2]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
from ipywidgets import interact, IntSlider, Dropdown
from scipy.linalg import orth
from scipy.linalg import norm
from ROVir import form_virtual_coil_data, rovir
from raw_deface_opt import set_masks, make_A_B
import numpy as np
from IPython.display import clear_output, display
import nibabel as nib

# make slider for selecting top virtual coils
n_coils_slider = IntSlider(value=1, min=1, 
                    max=nc, step=1, 
                    description='Number of Top Virtual Coils', 
                    continuous_update=False, 
                    style={'description_width': 'initial'},
                    layout={'width': '600px'})

# make sliders for x, y, z slices of the brain
x_slicer = IntSlider(value=data.shape[0]//2, min=0,
                     max = data.shape[0]-1, step=1,
                     description='Sagittal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

y_slicer = IntSlider(value=data.shape[1]//2, min=0,
                     max = data.shape[1]-1, step=1,
                     description='Coronal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

z_slicer = IntSlider(value=data.shape[2]//2, min=0,
                     max = data.shape[2]-1, step=1,
                     description='Axial Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

# make drop down box to select masking options
mask_selector = Dropdown(options=['Default', 'A', 'B', 'C', 'D', 'E'],
                         value='Default', description='Masking Scheme',
                         style={'description_width': 'initial'},
                         layout={'width': '300px'})

# make slider to choose gap between brain and face masks
gap_slider = IntSlider(value=10, min=0, max=30, 
                       step=1, description='Brain and Face Mask Gap',
                       continuous_update=False, 
                       style={'description_width': 'initial'},
                       layout={'width': '600px'})


In [4]:
@interact
def coil_retention_slider(n_coils=n_coils_slider, 
                          x=x_slicer, y=y_slicer, z=z_slicer, 
                          mask_option=mask_selector,
                          gap=gap_slider):
    clear_output()
    display('Recomputing defaced images...') # loading message
    
    # load default face and brain mask
    maskA = (nib.load(f'segmentations/output_mask_{dataID}/brain.nii.gz')).get_fdata()
    maskB = (nib.load(f'segmentations/output_mask_{dataID}/face.nii.gz')).get_fdata()

    if mask_option != 'Default':
        maskA, maskB = set_masks(maskA, maskB, mask_option, gap) # compute manipulated masks
    
    brain_covar, face_covar = make_A_B(og_image, nc, maskA, maskB) # compute covariance matrices

    eigenvec = rovir(nc, brain_covar, face_covar) # compute eigenvectors
    
    eigenvec_retain = orth(eigenvec[:, :n_coils]) # retain n_coils eigenvectors
    virtual_coil_data = form_virtual_coil_data(eigenvec_retain, data) # form the virtual coil data with the top eigenvectors

    # compute the image data from the virtual coil data 
    image = np.fft.fftshift(np.fft.ifftn(np.fft.fftshift(virtual_coil_data, axes = (0, 1, 2)), axes=(0, 1, 2)), axes = (0, 1, 2))
    image_rsos = np.sqrt(np.sum(np.abs(image)**2, axis=3))
    
    # calculate the percentage of signal retained from the brain and face regions
    orth_proj = eigenvec_retain @ eigenvec_retain.conj().T
    brain_retain = (norm((orth_proj @ brain_covar @ orth_proj), ord = 'fro') / norm (brain_covar, ord = 'fro'))*100
    face_retain = (norm((orth_proj @ face_covar @ orth_proj), ord = 'fro') / norm (face_covar, ord = 'fro'))*100
    
    clear_output()
    
    # plot original, mask overlayed, and defaced images
    plt.figure(figsize=(15, 10))

    plt.subplot(3,3,1)
    plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')

    plt.subplot(3,3,2)
    plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')

    plt.subplot(3,3,3)
    plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')

    plt.subplot(3,3,4)
    plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')
    plt.imshow(np.rot90(maskB[:, :, z]), alpha = 0.3, cmap = 'Reds')
    plt.imshow(np.rot90(maskA[:, :, z]), alpha = 0.3, cmap = 'Greens')
    
    plt.subplot(3,3,5)
    plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')
    plt.imshow(np.rot90(maskB[x, :, :]), alpha = 0.3, cmap = 'Reds')
    plt.imshow(np.rot90(maskA[x, :, :]), alpha = 0.3, cmap = 'Greens')
    
    plt.subplot(3,3,6)
    plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')
    plt.imshow(np.rot90(maskB[:, y, :]), alpha = 0.3, cmap = 'Reds')
    plt.imshow(np.rot90(maskA[:, y, :]), alpha = 0.3, cmap = 'Greens')

    plt.subplot(3,3,7)
    plt.imshow(np.rot90(image_rsos[:, :, z]), cmap='gray')

    plt.subplot(3,3,8)
    plt.imshow(np.rot90(image_rsos[x, :, :]), cmap='gray')

    plt.subplot(3,3,9)
    plt.imshow(np.rot90(image_rsos[:, y, :]), cmap='gray')

    plt.show()

    print(f'Brain region signal retention: {brain_retain}')
    print(f'Face region signal retention: {face_retain}')


interactive(children=(IntSlider(value=1, continuous_update=False, description='Number of Top Virtual Coils', l…